# Do-calculus engine demo

Run the engine (examples/new/do_calculus.py) on confounder, instrument, and nonID
examples. Software tests are in `tests/`.

In [ ]:
import do_calculus as dc
from hierarchicalcausalmodels.models import HSCMParametric

def _empty_fun(*args, **kwargs):
    return None

In [ ]:
if not dc.PYAGNUM_AVAILABLE:
    print("pyagrum not installed; skip do-calculus demos.")
else:
    # Confounder
    h_conf = HSCMParametric(
        nodes={"U", "A", "Y"}, edges={("U", "A"), ("U", "Y"), ("A", "Y")},
        unit_nodes={"U"}, subunit_nodes={"A", "Y"},
        sizes=[3], node_functions={"U": _empty_fun, "A": _empty_fun, "Y": _empty_fun}, data={},
    )
    conf_cgm = dc.collapse(h_conf)
    aug_conf = dc.augment_collapsed_model(conf_cgm, "Q^y", {"Q^{y|a}", "Q^a"})
    aug_conf.unobserved_variables = {"U"}
    res_conf = dc.identify_effect(aug_conf, Y="Q^y", X="Q^a", unobserved={"U"})
    print("Confounder: identifiable =", res_conf.identifiable)
    if res_conf.formula_latex:
        print("  Formula:", res_conf.formula_latex[:80], "...")

In [ ]:
if dc.PYAGNUM_AVAILABLE:
    # Instrument
    h_inst = HSCMParametric(
        nodes={"U", "Y", "Z", "A"}, edges={("U", "A"), ("U", "Y"), ("Z", "A"), ("A", "Y")},
        unit_nodes={"U", "Y"}, subunit_nodes={"Z", "A"},
        sizes=[3], node_functions={n: _empty_fun for n in ["U", "Y", "Z", "A"]}, data={},
    )
    inst_cgm = dc.collapse(h_inst)
    inst_cgm = dc.augment_collapsed_model(inst_cgm, "Q^a", {"Q^z", "Q^{a|z}"})
    inst_cgm = dc.marginalize_augmented_model(inst_cgm, "Y", {"Q^z"})
    inst_cgm.unobserved_variables = {"U"}
    res_inst = dc.identify_effect(inst_cgm, Y="Y", X="Q^a", unobserved={"U"})
    print("Instrument: identifiable =", res_inst.identifiable)
    if res_inst.formula_latex:
        print("  Formula:", res_inst.formula_latex[:80], "...")

In [ ]:
if dc.PYAGNUM_AVAILABLE:
    # NonID
    h_nonid = HSCMParametric(
        nodes={"U", "A", "W", "Y"}, edges={("U", "A"), ("U", "W"), ("A", "W"), ("A", "Y"), ("W", "Y")},
        unit_nodes={"U", "W"}, subunit_nodes={"A", "Y"},
        sizes=[3], node_functions={n: _empty_fun for n in ["U", "A", "W", "Y"]}, data={},
    )
    nonid_cgm = dc.collapse(h_nonid)
    nonid_aug = dc.augment_collapsed_model(nonid_cgm, "Q^y", {"Q^a", "Q^{y|a}"})
    nonid_aug.unobserved_variables = {"U"}
    res_nonid = dc.identify_effect(nonid_aug, Y="Q^y", X="Q^a", unobserved={"U"})
    print("nonID_ex1: identifiable =", res_nonid.identifiable)
    if res_nonid.error:
        print("  Error:", res_nonid.error[:60], "...")